# 🚗 NHTSA Vehicle Safety Recall Intelligence (2010–2026)
## Complete EDA, Risk Analysis & NLP on 47,707 US Recalls

**47,707 recalls · 789 makes · 16 years · ML severity scores · Takata deep-dive**

> *"Safety doesn't happen by accident."* — Unknown

### What This Notebook Covers
1. Dataset Overview & Data Quality
2. Annual Recall Volume & Trends
3. Defect Severity & Risk Tier Distribution
4. Top Makes & Manufacturers
5. Component Category Analysis
6. Fire Risk Intelligence
7. Recall Scale & Units Affected
8. Notification Lag Analysis
9. Software / EV Recall Trend (Post-2020 Rise)
10. Takata Airbag Crisis Deep-Dive
11. Make Origin Comparison (US vs Japanese vs German vs EV-First)
12. NLP: TF-IDF on Defect Summaries
13. Risk Score Predictor (ML Model)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 110

# Brand palette
RED    = '#E74C3C'
ORANGE = '#E67E22'
GOLD   = '#F39C12'
GREEN  = '#27AE60'
BLUE   = '#2980B9'
NAVY   = '#1A252F'
TEAL   = '#16A085'
PURPLE = '#8E44AD'
GRAY   = '#7F8C8D'

sev_palette   = {'CRITICAL': RED, 'HIGH': ORANGE, 'MEDIUM': GOLD, 'LOW': GREEN}
tier_palette  = {'HIGH': RED, 'MODERATE': ORANGE, 'LOW': GREEN}
scale_palette = {'MEGA': '#6C3483', 'MASSIVE': RED, 'LARGE': ORANGE, 'MEDIUM': GOLD, 'SMALL': GREEN}

print("✅ Libraries loaded")

## 1. Dataset Overview & Data Quality

In [ ]:
df = pd.read_csv("/kaggle/input/nhtsa-vehicle-safety-recall-intelligence-2010-2026/nhtsa_vehicle_safety_recall_intelligence_ultimate.csv", low_memory=False)

print(f"Shape:          {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Recall years:   {df['recall_year'].min()} – {df['recall_year'].max()}")
print(f"Unique makes:   {df['vehicle_make'].nunique():,}")
print(f"Unique models:  {df['vehicle_model'].nunique():,}")
print(f"Manufacturers:  {df['manufacturer_name'].nunique():,}")
print(f"Total units:    {df['units_affected_num'].sum() / 1e9:.2f} billion")
print(f"Fire-risk recs: {df['fire_risk_flag'].sum():,}")
print(f"\nMissing values (key columns):")
key_cols = ['defect_severity','fire_risk_flag','recall_risk_score','risk_tier',
            'component_category','days_to_owner_notification','units_affected_num']
print(df[key_cols].isnull().sum().to_string())

In [ ]:
df.head(3)

## 2. Annual Recall Volume & Trends (2010–2026)

In [ ]:
annual = df.groupby('recall_year').agg(
    recalls=('record_id', 'count'),
    units_billion=('units_affected_num', lambda x: x.sum()/1e9),
    critical=('defect_severity', lambda x: (x=='CRITICAL').sum()),
    fire_risk=('fire_risk_flag', 'sum'),
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].bar(annual['recall_year'], annual['recalls'],
              color=[RED if y >= 2020 else BLUE for y in annual['recall_year']], edgecolor='white', linewidth=0.4)
axes[0,0].set_title('Annual Recall Record Count', fontsize=13, fontweight='bold')
axes[0,0].set_ylabel('Recall Records')
axes[0,0].tick_params(axis='x', rotation=45)
for bar, val in zip(axes[0,0].patches, annual['recalls']):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30, f'{val:,}',
                   ha='center', fontsize=7, fontweight='bold')

axes[0,1].bar(annual['recall_year'], annual['units_billion'],
              color=ORANGE, edgecolor='white', linewidth=0.4, alpha=0.9)
axes[0,1].set_title('Total Units Recalled per Year (Billions)', fontsize=13, fontweight='bold')
axes[0,1].set_ylabel('Units (Billions)')
axes[0,1].tick_params(axis='x', rotation=45)

axes[1,0].plot(annual['recall_year'], annual['critical'], marker='o', color=RED,
               linewidth=2.2, label='CRITICAL')
axes[1,0].fill_between(annual['recall_year'], annual['critical'], alpha=0.12, color=RED)
axes[1,0].set_title('CRITICAL Severity Recalls per Year', fontsize=13, fontweight='bold')
axes[1,0].set_ylabel('Count'); axes[1,0].tick_params(axis='x', rotation=45)

# Severity mix over time
sev_yr = df.groupby(['recall_year','defect_severity']).size().unstack(fill_value=0)
sev_yr[['CRITICAL','HIGH','MEDIUM','LOW']].plot.area(
    ax=axes[1,1], color=[RED,ORANGE,GOLD,GREEN], alpha=0.8, linewidth=0)
axes[1,1].set_title('Severity Mix Over Time', fontsize=13, fontweight='bold')
axes[1,1].set_ylabel('Records'); axes[1,1].tick_params(axis='x', rotation=45)
axes[1,1].legend(fontsize=8)

plt.suptitle('NHTSA Recall Trends 2010–2026', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 3. Defect Severity & Risk Tier Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Severity donut
sev_counts = df['defect_severity'].value_counts().reindex(['CRITICAL','HIGH','MEDIUM','LOW'])
wedges, texts, autotexts = axes[0].pie(
    sev_counts, labels=sev_counts.index, autopct='%1.1f%%',
    colors=[RED, ORANGE, GOLD, GREEN], startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}, pctdistance=0.8)
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')
centre = plt.Circle((0,0), 0.55, color='white')
axes[0].add_artist(centre)
axes[0].text(0, 0, f"{len(df):,}\nRecalls", ha='center', va='center', fontsize=10, fontweight='bold')
axes[0].set_title('ML Defect Severity Distribution', fontsize=13, fontweight='bold')

# Risk tier
tier_counts = df['risk_tier'].value_counts().reindex(['HIGH','MODERATE','LOW'])
axes[1].bar(tier_counts.index, tier_counts.values,
            color=[RED, ORANGE, GREEN], edgecolor='white', linewidth=0.8, width=0.5)
axes[1].set_title('Composite Risk Tier', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Recall Records')
for bar, val in zip(axes[1].patches, tier_counts.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+80, f'{val:,}',
                 ha='center', fontsize=11, fontweight='bold')

# Risk score distribution
df['recall_risk_score'].plot.hist(bins=40, ax=axes[2], color=BLUE, edgecolor='white', alpha=0.85)
axes[2].axvline(df['recall_risk_score'].mean(), color=RED, linewidth=2.2, linestyle='--',
                label=f'Mean: {df["recall_risk_score"].mean():.1f}')
axes[2].set_title('Composite Risk Score Distribution (0–100)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Risk Score'); axes[2].legend()

plt.tight_layout(); plt.show()

print(f"Severity breakdown:")
for sev, cnt in sev_counts.items():
    print(f"  {sev:8s}: {cnt:6,} ({cnt/len(df)*100:.1f}%)")

In [ ]:
# Cross-tab: severity vs risk tier
ct = pd.crosstab(df['defect_severity'], df['risk_tier'],
                 normalize='index').reindex(['CRITICAL','HIGH','MEDIUM','LOW']) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ct[['HIGH','MODERATE','LOW']].plot.bar(
    ax=ax, color=[RED, ORANGE, GREEN], edgecolor='white', linewidth=0.4)
ax.set_title('Risk Tier by Defect Severity (% within severity level)', fontsize=13, fontweight='bold')
ax.set_ylabel('%'); ax.set_xlabel('Defect Severity')
ax.tick_params(axis='x', rotation=0); ax.legend(title='Risk Tier', fontsize=9)
plt.tight_layout(); plt.show()

## 4. Top Makes & Manufacturers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

top25_makes = df.groupby('vehicle_make').agg(
    recalls=('record_id','count'),
    critical=('defect_severity', lambda x:(x=='CRITICAL').sum()),
    units_m=('units_affected_num', lambda x: x.sum()/1e6),
    avg_risk=('recall_risk_score','mean')
).nlargest(25,'recalls').sort_values('recalls')

bars = axes[0].barh(top25_makes.index, top25_makes['recalls'],
                    color=BLUE, edgecolor='white', linewidth=0.4, alpha=0.85)
axes[0].barh(top25_makes.index, top25_makes['critical'],
             color=RED, edgecolor='white', linewidth=0.4, alpha=0.85, label='CRITICAL')
axes[0].set_title('Top 25 Makes by Recall Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Records')
axes[0].legend(handles=[mpatches.Patch(color=BLUE,label='Total'),
                         mpatches.Patch(color=RED,label='CRITICAL')], fontsize=9)

# Avg risk by top 20 manufacturers
top20_mfr = df.groupby('manufacturer_name').agg(
    recalls=('record_id','count'),
    avg_risk=('recall_risk_score','mean'),
    fire_risk=('fire_risk_flag','sum')
).nlargest(20,'recalls').sort_values('avg_risk')

colors_mfr = [RED if v > 40 else ORANGE if v > 30 else GREEN for v in top20_mfr['avg_risk']]
axes[1].barh(top20_mfr.index, top20_mfr['avg_risk'], color=colors_mfr, edgecolor='white', linewidth=0.4)
axes[1].axvline(df['recall_risk_score'].mean(), color='black', linestyle='--', alpha=0.5, label='Global Mean')
axes[1].set_title('Avg Risk Score — Top 20 Manufacturers by Volume', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Average Composite Risk Score'); axes[1].legend()
for bar, (mfr, row) in zip(axes[1].patches, top20_mfr.iterrows()):
    axes[1].text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                 f'{row["recalls"]:,} recalls', va='center', fontsize=7)

plt.tight_layout(); plt.show()

## 5. Component Category Analysis

In [ ]:
comp_stats = df.groupby('component_category').agg(
    recalls=('record_id','count'),
    units_m=('units_affected_num', lambda x: x.sum()/1e6),
    critical_pct=('defect_severity', lambda x: (x=='CRITICAL').mean()*100),
    fire_risk=('fire_risk_flag','sum'),
    avg_risk=('recall_risk_score','mean')
).sort_values('recalls', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

palette_comp = sns.color_palette("husl", len(comp_stats))
comp_stats['recalls'].sort_values().plot.barh(
    ax=axes[0], color=palette_comp, edgecolor='white', linewidth=0.4)
axes[0].set_title('Recall Count by Component Category', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Recall Records')

# CRITICAL % by component
comp_stats['critical_pct'].sort_values().plot.barh(
    ax=axes[1],
    color=[RED if v > 25 else ORANGE if v > 20 else GOLD for v in comp_stats['critical_pct'].sort_values()],
    edgecolor='white', linewidth=0.4)
axes[1].axvline(comp_stats['critical_pct'].mean(), color='black', linestyle='--', alpha=0.5, label=f'Mean: {comp_stats["critical_pct"].mean():.1f}%')
axes[1].set_title('% CRITICAL Severity by Component', fontsize=13, fontweight='bold')
axes[1].set_xlabel('% CRITICAL'); axes[1].legend()

plt.tight_layout(); plt.show()

print(comp_stats[['recalls','units_m','critical_pct','fire_risk','avg_risk']].round(1).to_string())

## 6. 🔥 Fire Risk Intelligence

In [ ]:
fire = df[df['fire_risk_flag']==1].copy()
no_fire = df[df['fire_risk_flag']==0].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Fire risk by make
fire_by_make = fire['vehicle_make'].value_counts().head(15).sort_values()
fire_by_make.plot.barh(ax=axes[0], color=RED, edgecolor='white', linewidth=0.4)
axes[0].set_title('🔥 Top 15 Makes with Fire-Risk Recalls', fontsize=12, fontweight='bold')

# Fire risk by component
fire_comp = fire['component_category'].value_counts().sort_values()
fire_comp.plot.barh(ax=axes[1], color=ORANGE, edgecolor='white', linewidth=0.4)
axes[1].set_title('Fire-Risk Recalls by Component Category', fontsize=12, fontweight='bold')

# Fire vs non-fire severity
sev_compare = pd.DataFrame({
    'Fire Risk': fire['defect_severity'].value_counts(normalize=True)*100,
    'No Fire Risk': no_fire['defect_severity'].value_counts(normalize=True)*100
}).reindex(['CRITICAL','HIGH','MEDIUM','LOW']).fillna(0)
sev_compare.plot.bar(ax=axes[2], color=[RED, BLUE], edgecolor='white', linewidth=0.4)
axes[2].set_title('Severity: Fire Risk vs Non-Fire', fontsize=12, fontweight='bold')
axes[2].set_ylabel('%'); axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f"Fire-risk recalls: {len(fire):,} ({len(fire)/len(df)*100:.2f}% of total)")
print(f"Fire risk avg units affected: {fire['units_affected_num'].mean():,.0f}")
print(f"Non-fire avg units affected:  {no_fire['units_affected_num'].mean():,.0f}")
print(f"\nTop fire-risk years:")
print(fire['recall_year'].value_counts().head(8).to_string())

## 7. Recall Scale & Units Affected

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scale_order = ['MEGA','MASSIVE','LARGE','MEDIUM','SMALL']
scale_counts = df['recall_scale'].value_counts().reindex(scale_order)
axes[0].bar(scale_counts.index, scale_counts.values,
            color=['#6C3483',RED,ORANGE,GOLD,GREEN], edgecolor='white', linewidth=0.5)
axes[0].set_title('Recall Scale Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(axes[0].patches, scale_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+100, f'{val:,}',
                 ha='center', fontsize=9, fontweight='bold')

np.log10(df['units_affected_num'].clip(lower=1)).plot.hist(
    bins=40, ax=axes[1], color=BLUE, edgecolor='white', alpha=0.85)
axes[1].set_title('Units Affected Distribution (log10)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log10(Units Affected)')

for val, label in [(3,'1K'), (4,'10K'), (5,'100K'), (6,'1M'), (7,'10M')]:
    axes[1].axvline(val, color='gray', linestyle=':', alpha=0.5)
    axes[1].text(val+0.05, axes[1].get_ylim()[1]*0.9, label, fontsize=8, color='gray')

plt.tight_layout(); plt.show()

print(f"Recall scale statistics:")
print(df.groupby('recall_scale')['units_affected_num'].agg(['mean','median','sum']).reindex(scale_order).applymap(lambda x: f'{x:,.0f}').to_string())

## 8. Notification Lag Analysis — How Long Do Owners Wait?

In [ ]:
lag = df.dropna(subset=['days_to_owner_notification']).copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

lag['days_to_owner_notification'].clip(upper=400).plot.hist(
    bins=50, ax=axes[0], color=TEAL, edgecolor='white', alpha=0.85)
axes[0].axvline(lag['days_to_owner_notification'].median(), color=RED, linewidth=2, linestyle='--',
                label=f'Median: {lag["days_to_owner_notification"].median():.0f} days')
axes[0].set_title('Owner Notification Lag Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Days (capped at 400)'); axes[0].legend()

# Lag by severity
sev_order = ['CRITICAL','HIGH','MEDIUM','LOW']
lag_sev = lag.groupby('defect_severity')['days_to_owner_notification'].median().reindex(sev_order)
axes[1].bar(lag_sev.index, lag_sev.values,
            color=[RED,ORANGE,GOLD,GREEN], edgecolor='white', linewidth=0.5)
axes[1].set_title('Median Notification Lag by Severity', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Median Days')
for bar, val in zip(axes[1].patches, lag_sev.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{val:.0f}d',
                 ha='center', fontsize=11, fontweight='bold')

# Lag trend over years
lag.groupby('recall_year')['days_to_owner_notification'].median().plot(
    ax=axes[2], marker='o', color=TEAL, linewidth=2)
axes[2].set_title('Median Notification Lag by Year', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Median Days'); axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

print(f"Notification lag statistics (days):")
print(f"  Mean:   {lag['days_to_owner_notification'].mean():.1f}")
print(f"  Median: {lag['days_to_owner_notification'].median():.1f}")
print(f"  90th %: {lag['days_to_owner_notification'].quantile(0.9):.1f}")
print(f"  Max:    {lag['days_to_owner_notification'].max():.0f}")
print(f"\nRecalls with >180 day lag: {(lag['days_to_owner_notification']>180).sum():,} ({(lag['days_to_owner_notification']>180).mean()*100:.1f}%)")

## 9. 💻 Software / EV Recall Trend — The New Frontier

In [ ]:
sw = df[df['component_category']=='Software / EV'].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Growth over time
sw_yr = sw.groupby('recall_year').size()
all_yr = df.groupby('recall_year').size()
sw_pct = (sw_yr / all_yr * 100).fillna(0)

ax2 = axes[0].twinx()
axes[0].bar(sw_yr.index, sw_yr.values, color=PURPLE, edgecolor='white', alpha=0.8, label='SW/EV Recalls')
ax2.plot(sw_pct.index, sw_pct.values, marker='o', color=RED, linewidth=2, label='% of Total')
axes[0].set_title('Software/EV Recalls per Year', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count', color=PURPLE); ax2.set_ylabel('% of Total', color=RED)
axes[0].tick_params(axis='x', rotation=45)

# Top makes for SW/EV
sw['vehicle_make'].value_counts().head(12).sort_values().plot.barh(
    ax=axes[1], color=PURPLE, edgecolor='white', linewidth=0.4, alpha=0.85)
axes[1].set_title('Top Makes for Software/EV Recalls', fontsize=12, fontweight='bold')

# Severity profile SW/EV vs others
sw_sev = sw['defect_severity'].value_counts(normalize=True)*100
other_sev = df[df['component_category']!='Software / EV']['defect_severity'].value_counts(normalize=True)*100
sev_comp = pd.DataFrame({'Software/EV': sw_sev, 'All Others': other_sev}).reindex(['CRITICAL','HIGH','MEDIUM','LOW']).fillna(0)
sev_comp.plot.bar(ax=axes[2], color=[PURPLE, BLUE], edgecolor='white', linewidth=0.4)
axes[2].set_title('Severity: Software/EV vs All Others', fontsize=12, fontweight='bold')
axes[2].set_ylabel('%'); axes[2].tick_params(axis='x', rotation=0); axes[2].legend()

plt.tight_layout(); plt.show()

print(f"Software/EV recalls: {len(sw):,} total")
print(f"Years with SW/EV recalls: {sw['recall_year'].min()}–{sw['recall_year'].max()}")
print(f"Pre-2020 SW/EV recalls: {(sw['recall_year']<2020).sum()}")
print(f"Post-2020 SW/EV recalls: {(sw['recall_year']>=2020).sum()}")
print(f"\nTop 5 SW/EV makes:")
print(sw['vehicle_make'].value_counts().head(5).to_string())

## 10. 💥 Takata Airbag Crisis Deep-Dive
The Takata airbag crisis is the **largest automotive safety recall in history** — affecting over 100 million vehicles worldwide. The inflators could rupture and send metal shrapnel into the cabin.

In [ ]:
takata = df[df['defect_summary'].str.contains('takata|TAKATA|Takata', na=False, case=False)].copy()
print(f"Takata-related records: {len(takata):,}")
print(f"Total Takata units: {takata['units_affected_num'].sum() / 1e6:.1f} million")
print(f"Takata makes: {takata['vehicle_make'].nunique()}")
print(f"Year range: {takata['recall_year'].min()}–{takata['recall_year'].max()}")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

takata.groupby('recall_year').size().plot.bar(
    ax=axes[0,0], color=RED, edgecolor='white', linewidth=0.4, alpha=0.9)
axes[0,0].set_title('Takata Recall Records per Year', fontsize=12, fontweight='bold')
axes[0,0].tick_params(axis='x', rotation=45)

takata.groupby('recall_year')['units_affected_num'].sum().div(1e6).plot.bar(
    ax=axes[0,1], color=ORANGE, edgecolor='white', linewidth=0.4, alpha=0.9)
axes[0,1].set_title('Takata Units Recalled (Millions) per Year', fontsize=12, fontweight='bold')
axes[0,1].set_ylabel('Million Units'); axes[0,1].tick_params(axis='x', rotation=45)

takata['vehicle_make'].value_counts().head(15).sort_values().plot.barh(
    ax=axes[1,0], color=RED, edgecolor='white', linewidth=0.4, alpha=0.85)
axes[1,0].set_title('Top 15 Makes Affected by Takata', fontsize=12, fontweight='bold')

takata['defect_severity'].value_counts().reindex(['CRITICAL','HIGH','MEDIUM','LOW']).plot.bar(
    ax=axes[1,1], color=[RED,ORANGE,GOLD,GREEN], edgecolor='white', linewidth=0.4)
axes[1,1].set_title('Takata Recall Severity Distribution', fontsize=12, fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=0)

plt.suptitle('Takata Airbag Crisis — Largest Automotive Recall in History', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

## 11. Make Origin Comparison — US vs Japanese vs German vs EV-First

In [ ]:
origins = df[df['make_origin'].isin(['US Domestic','Japanese','German','EV-First Brand'])].copy()
origin_order = ['US Domestic','Japanese','German','EV-First Brand']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Risk score by origin
sns.boxplot(data=origins, x='make_origin', y='recall_risk_score',
            order=origin_order, ax=axes[0],
            palette={'US Domestic':BLUE,'Japanese':RED,'German':GOLD,'EV-First Brand':PURPLE})
axes[0].set_title('Risk Score by Make Origin', fontsize=12, fontweight='bold')
axes[0].set_xlabel(''); axes[0].tick_params(axis='x', rotation=15)

# Notification lag by origin
lag_orig = origins.dropna(subset=['days_to_owner_notification'])
sns.boxplot(data=lag_orig, x='make_origin', y='days_to_owner_notification',
            order=origin_order, showfliers=False, ax=axes[1],
            palette={'US Domestic':BLUE,'Japanese':RED,'German':GOLD,'EV-First Brand':PURPLE})
axes[1].set_title('Notification Lag by Make Origin', fontsize=12, fontweight='bold')
axes[1].set_xlabel(''); axes[1].tick_params(axis='x', rotation=15)

# CRITICAL % by origin
crit_pct = origins.groupby('make_origin').apply(
    lambda x: (x['defect_severity']=='CRITICAL').mean()*100).reindex(origin_order)
axes[2].bar(crit_pct.index, crit_pct.values,
            color=[BLUE, RED, GOLD, PURPLE], edgecolor='white', linewidth=0.5, width=0.5)
axes[2].set_title('CRITICAL Recall % by Make Origin', fontsize=12, fontweight='bold')
axes[2].set_ylabel('%')
for bar, val in zip(axes[2].patches, crit_pct.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{val:.1f}%',
                 ha='center', fontsize=11, fontweight='bold')

plt.tight_layout(); plt.show()

## 12. NLP: TF-IDF Analysis of Defect Summaries

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Clean defect summaries
corpus = df['defect_summary'].fillna('').str.lower()

# Per-severity top terms
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sev_levels = ['CRITICAL','HIGH','MEDIUM','LOW']
colors_nlp = [RED, ORANGE, GOLD, GREEN]

for ax, sev, col in zip(axes.flatten(), sev_levels, colors_nlp):
    sev_corpus = corpus[df['defect_severity']==sev]
    tfidf = TfidfVectorizer(max_features=500, stop_words='english', min_df=3, ngram_range=(1,2))
    tfidf.fit_transform(sev_corpus)
    top_terms = pd.Series(tfidf.idf_, index=tfidf.get_feature_names_out()).nsmallest(15)
    top_terms = top_terms.sort_values(ascending=True)
    ax.barh(top_terms.index, 1/top_terms.values, color=col, edgecolor='white', linewidth=0.4, alpha=0.85)
    ax.set_title(f'{sev} — Top 15 Terms (TF-IDF)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Relative Frequency (1/IDF)')

plt.suptitle('Most Distinctive Terms per Defect Severity (TF-IDF)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Top bigrams across fire-risk recalls
fire_corpus = corpus[df['fire_risk_flag']==1]
tfidf_fire = TfidfVectorizer(max_features=1000, stop_words='english', min_df=2, ngram_range=(2,3))
tfidf_fire.fit_transform(fire_corpus)
top_fire_terms = pd.Series(tfidf_fire.idf_, index=tfidf_fire.get_feature_names_out()).nsmallest(20)

fig, ax = plt.subplots(figsize=(10, 6))
top_fire_terms.sort_values().apply(lambda x: 1/x).sort_values().plot.barh(
    ax=ax, color=RED, edgecolor='white', linewidth=0.4, alpha=0.85)
ax.set_title('Top 20 Bigrams/Trigrams in Fire-Risk Defect Descriptions', fontsize=13, fontweight='bold')
ax.set_xlabel('Relative Frequency (1/IDF)')
plt.tight_layout(); plt.show()

## 13. Risk Tier Prediction Model

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

model_df = df.copy()
for col in ['component_category','recall_scale','make_origin','vehicle_era','defect_severity']:
    model_df[col+'_enc'] = LabelEncoder().fit_transform(model_df[col].fillna('Unknown').astype(str))

le_target = LabelEncoder()
model_df['risk_tier_enc'] = le_target.fit_transform(model_df['risk_tier'])

features = ['defect_severity_enc','component_category_enc','recall_scale_enc',
            'make_origin_enc','fire_risk_flag','manufacturer_recall_count',
            'is_repeat_manufacturer','units_affected_num','recall_year',
            'vehicle_era_enc','defect_severity_score']

X = model_df[features].fillna(0).values
y = model_df['risk_tier_enc'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in [('Random Forest', RandomForestClassifier(200, class_weight='balanced', random_state=42, n_jobs=-1)),
                   ('Gradient Boosting', GradientBoostingClassifier(150, max_depth=4, random_state=42))]:
    acc = cross_val_score(clf, X, y, cv=skf, scoring='accuracy')
    f1  = cross_val_score(clf, X, y, cv=skf, scoring='f1_macro')
    print(f"{name:25s}  Accuracy={acc.mean():.4f}±{acc.std():.4f}  F1-macro={f1.mean():.4f}")

In [ ]:
# Feature importance
gb = GradientBoostingClassifier(150, max_depth=4, random_state=42)
gb.fit(X, y)

fi = pd.Series(gb.feature_importances_, index=features).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

fi.plot.barh(ax=axes[0], color=[RED if v>0.1 else BLUE for v in fi.values], edgecolor='white')
axes[0].set_title('Feature Importance — Risk Tier Classifier', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Relative Importance')

# Confusion matrix
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import ConfusionMatrixDisplay
y_pred = cross_val_predict(gb, X, y, cv=3)
ConfusionMatrixDisplay.from_predictions(
    le_target.inverse_transform(y), le_target.inverse_transform(y_pred),
    ax=axes[1], colorbar=False, cmap='Blues',
    display_labels=le_target.classes_)
axes[1].set_title('Confusion Matrix (3-Fold CV)', fontsize=13, fontweight='bold')

plt.tight_layout(); plt.show()

## 📋 Key Findings

### 📊 Scale
- **47,707 recall records** across 789 makes covering 2010–2026
- **12.3 billion total units** recalled — almost 1.5× the world's population
- **~25% CRITICAL severity** — one in four recalls is the most dangerous classification

### 🔥 Fire Risk
- **206 fire-risk recalls** — small in number but disproportionately critical
- **Fuel System and Electrical** components account for >80% of fire-risk records
- Fire-risk recalls have **3× higher CRITICAL rate** than non-fire recalls

### ⏱️ Notification Lag
- Median owner notification: **50 days** after recall initiation
- CRITICAL recalls don't always get faster notifications — median lag similar across severity
- Some owners waited **>7 years** (2,675 days maximum)

### 💻 Software/EV Revolution
- **Zero** SW/EV recalls before 2020 → **275 records** by 2026
- Tesla, Rivian, Lucid, and traditional EV-converted brands dominating this category
- SW/EV recalls tend to be higher scale (OTA updates can cover many units) but lower injury risk

### 💥 Takata
- **Fully captured** across all affected brands — Honda, Toyota, Ford, BMW, and 60+ more
- Peak years: **2016–2017** with millions of units per year
- Still ongoing as of 2026 with replacement backlog

### 🤖 ML Model
- Risk tier predictable at **>85% accuracy** — severity score and units affected are top predictors
- `defect_severity_score` (transformer embedding) is the single strongest feature

---
*Data source: NHTSA ODI Flat File + engineered ML columns*  
*If this notebook was useful, please upvote! 🙏*